# 02 — Feature Engineering: Pre-Earnings Technical Indicators

For each earnings event we look back at the **14 trading days** immediately before the
announcement ($T_{-14}$ to $T_{-1}$) and turn the price action into model features.

**Key design rules:**
1. **$T_{-14}..T_{-1}$ are trading days, not calendar days** — we slice by row position in the sorted series, not by date math.
2. **Indicators are computed on the FULL price history first, then sliced** — so RSI(14), SMA(20), MACD etc. are fully warmed up across the whole window (no leading NaNs).
3. **$T_{-1}$ is the last trading day STRICTLY before the announcement** — many firms report after the close, so the announcement-day bar can already contain the reaction. We exclude it to avoid leakage.

Indicators are computed with **TA-Lib** (`pip install TA-Lib`). Output: one feature row per earnings event, joined back to the target labels.

In [1]:
import os
import pandas as pd
import talib

# ── Config ──────────────────────────────────────────
DATA_DIR      = "../data/raw"
BASE_CSV      = os.path.join(DATA_DIR, "earnings_base.csv")
PRICE_DIR     = os.path.join(DATA_DIR, "ticker_prices")
PROCESSED_DIR = "../data/processed"
os.makedirs(PROCESSED_DIR, exist_ok=True)

LOOKBACK = 14   # trading days in the window: T-14 .. T-1

## Step 1 — Compute indicators on the full price series (TA-Lib)

Everything rolling/smoothed (RSI, SMA, MACD) is computed over the entire per-ticker history.
This is what guarantees the 14-day window is fully warmed up. TA-Lib works on NumPy arrays,
so we pass in the close/volume columns and assign the results back as new DataFrame columns.


In [2]:
def add_indicators(px: pd.DataFrame) -> pd.DataFrame:
    """Compute technical indicators on the FULL series (sorted ascending by date) via TA-Lib."""
    px = px.sort_values("date").reset_index(drop=True)
    close  = px["close"].to_numpy(dtype="float64")

   
    px["rsi_14"] = talib.RSI(close, timeperiod=14)

    #Simple Moving averages
    px["sma_10"] = talib.SMA(close, timeperiod=10)
    px["sma_20"] = talib.SMA(close, timeperiod=20)

    # Returns + realized volatility
    px["ret_1d"] = px["close"].pct_change()
    px["vol_20"] = px["ret_1d"].rolling(20).std()

    # MACD (12 / 26 / 9) — returns macd line, signal line, and histogram
    macd, macd_signal, _ = talib.MACD(close, fastperiod=12, slowperiod=26, signalperiod=9)
    px["macd"]        = macd
    px["macd_signal"] = macd_signal

    # Volume trend vs its 20-day average
    px["vol_ratio"] = px["volume"] / px["volume"].rolling(20).mean()
    return px

## Step 2 — Slice the $T_{-14}..T_{-1}$ window and aggregate into features

We find the last trading day strictly before the announcement ($T_{-1}$), take the 14 rows
ending there, and reduce them to a single feature row. We keep both a **point-in-time snapshot
at $T_{-1}$** and **trajectory features** aggregated across the window.

In [3]:
def window_features(px: pd.DataFrame, earnings_date) -> dict | None:
    """Slice T-14..T-1 by ROW POSITION and aggregate into one feature row."""
    earnings_date = pd.Timestamp(earnings_date)

    # last trading day STRICTLY before the announcement = T-1
    prior = px[px["date"] < earnings_date]
    if len(prior) < LOOKBACK:
        return None  # not enough history before this event

    win  = prior.iloc[-LOOKBACK:]   # the 14 rows: T-14 .. T-1
    last = win.iloc[-1]             # the T-1 row

    # guard against any residual NaN in the warm-up region
    if win[["rsi_14", "sma_20", "vol_20"]].isna().any().any():
        return None

    return {
        # ── point-in-time snapshot at T-1 ──
        "rsi_14_at_T1":    last["rsi_14"],
        "macd_hist_at_T1": last["macd"] - last["macd_signal"],
        "px_vs_sma20_T1":  last["close"] / last["sma_20"] - 1,
        "px_vs_sma10_T1":  last["close"] / last["sma_10"] - 1,

        # ── trajectory aggregated over the 14-day window ──
        "ret_14d":        win["close"].iloc[-1] / win["close"].iloc[0] - 1,
        "rsi_mean":       win["rsi_14"].mean(),
        "rsi_slope":      win["rsi_14"].iloc[-1] - win["rsi_14"].iloc[0],
        "vol_mean":       win["vol_20"].mean(),
        "vol_ratio_mean": win["vol_ratio"].mean(),
        "ret_std_14d":    win["ret_1d"].std(),
    }

## Step 3 — Historical earnings-surprise momentum

Independent of price data entirely: for each event, summarize that same ticker's own
beat/miss track record over its **prior** quarters. Many companies beat (or miss)
consensus repeatedly — managed guidance, conservative analyst modeling — which price
technicals can't see since they reset every quarter and know nothing about the company
itself. This is computed straight off `earnings_base.csv`, so it never touches
`PRICE_DIR` or TA-Lib.

Same T-1 discipline as the price window: every feature at row *i* is built only from
quarters strictly before *i* (via `.shift(1)` before rolling), so nothing leaks the
current quarter's own result.

In [4]:
SURPRISE_HISTORY_WINDOW = 4  # trailing quarters to summarize
SURPRISE_PCT_CAP = 3.0       # clip % surprise at +/-300% so a near-zero consensus_eps
                              # (e.g. a company forecast to roughly break even) can't blow
                              # up to +/-inf and poison the rolling average / StandardScaler


def surprise_history_features(g: pd.DataFrame, n: int = SURPRISE_HISTORY_WINDOW) -> pd.DataFrame:
    """Historical EPS-surprise momentum for one ticker's earnings history.

    A ticker's first quarter has no prior history, so these come out NaN and
    get dropped later — the same way events with insufficient price history do.
    """
    g = g.sort_values("earnings_date").reset_index(drop=True)
    surprise_pct = (
        (g["actual_eps"] - g["consensus_eps"]) / g["consensus_eps"].abs()
    ).clip(-SURPRISE_PCT_CAP, SURPRISE_PCT_CAP)
    prior_label = g["target_label"].shift(1)

    g["avg_surprise_pct_last4"] = surprise_pct.shift(1).rolling(n, min_periods=1).mean()
    g["beat_rate_last4"]        = prior_label.rolling(n, min_periods=1).mean()
    g["surprise_pct_last_q"]    = surprise_pct.shift(1)

    # consecutive-beat streak immediately before this quarter, reset on a miss
    run_id = (prior_label != prior_label.shift()).cumsum()
    streak = prior_label.groupby(run_id).cumcount() + 1
    g["beat_streak"] = streak.where(prior_label == 1, 0).astype("int64")

    return g

## Step 4 — Loop over earnings events and build the technical feature table

In [5]:
earnings = pd.read_csv(BASE_CSV, parse_dates=["earnings_date"])
print(f"Loaded {len(earnings)} earnings events for {earnings['ticker'].nunique()} tickers.")

base_cols = earnings.columns.tolist()
earnings = (
    earnings.groupby("ticker", group_keys=False)[earnings.columns]
    .apply(surprise_history_features)
)

price_cache = {}
rows = []
skipped = 0

for _, ev in earnings.iterrows():
    t = ev["ticker"]

    # load + compute indicators once per ticker, then reuse
    if t not in price_cache:
        path = os.path.join(PRICE_DIR, f"{t}_daily_prices.csv")
        if not os.path.exists(path):
            print(f"  -> No price file for {t}, skipping its events.")
            price_cache[t] = None
        else:
            px = pd.read_csv(path, parse_dates=["date"])
            price_cache[t] = add_indicators(px)

    px = price_cache[t]
    if px is None:
        skipped += 1
        continue

    feats = window_features(px, ev["earnings_date"])
    if feats is None:
        skipped += 1
        continue

    rows.append({"ticker": t, "earnings_date": ev["earnings_date"], **feats})

features_df = pd.DataFrame(rows)
print(f"Built features for {len(features_df)} events ({skipped} skipped for insufficient history).")
features_df.head()

Loaded 1187 earnings events for 60 tickers.


Built features for 1148 events (39 skipped for insufficient history).


,ticker,earnings_date,rsi_14_at_T1,macd_hist_at_T1,px_vs_sma20_T1,px_vs_sma10_T1,ret_14d,rsi_mean,rsi_slope,vol_mean,vol_ratio_mean,ret_std_14d
0,ABT,2021-10-20,49.379220,0.263563,0.008051,0.017823,0.014143,34.860095,19.933174,0.010334,1.024535,0.010192
1,ABT,2022-01-26,27.532126,-1.281694,-0.070017,-0.034338,-0.084753,38.516717,-21.898566,0.012201,1.236416,0.009417
2,ABT,2022-04-20,51.414768,-0.031037,0.007278,0.003988,0.000650,49.120066,1.094245,0.015706,0.942901,0.016564
3,ABT,2022-07-20,53.714133,0.282591,0.020945,0.018398,0.012086,48.178845,4.789194,0.017380,0.860467,0.015177
4,ABT,2022-10-19,57.838722,0.604707,0.043616,0.029627,0.073103,46.292190,26.335915,0.015388,1.061622,0.018445


## Step 5 — Join everything together and save

In [6]:
final = earnings.merge(features_df, on=["ticker", "earnings_date"], how="inner")

history_cols = ["avg_surprise_pct_last4", "beat_rate_last4", "surprise_pct_last_q", "beat_streak"]
no_history = final[history_cols].isna().any(axis=1)
if no_history.any():
    print(f"Dropping {no_history.sum()} events with no prior-quarter surprise history (ticker's first quarter in the dataset).")
    final = final[~no_history].reset_index(drop=True)

print(f"Final modeling table: {final.shape[0]} rows x {final.shape[1]} cols")
print("\nFeature columns:")
print([c for c in final.columns if c not in base_cols])

out_path = os.path.join(PROCESSED_DIR, "features_technical.csv")
final.to_csv(out_path, index=False)
print(f"\nSaved to: {out_path}")
final.head()

Dropping 22 events with no prior-quarter surprise history (ticker's first quarter in the dataset).
Final modeling table: 1126 rows x 22 cols

Feature columns:
['avg_surprise_pct_last4', 'beat_rate_last4', 'surprise_pct_last_q', 'beat_streak', 'rsi_14_at_T1', 'macd_hist_at_T1', 'px_vs_sma20_T1', 'px_vs_sma10_T1', 'ret_14d', 'rsi_mean', 'rsi_slope', 'vol_mean', 'vol_ratio_mean', 'ret_std_14d']

Saved to: ../data/processed/features_technical.csv


,ticker,earnings_date,actual_eps,consensus_eps,fiscal_date_ending,sector_group,surprise_amount,target_label,avg_surprise_pct_last4,beat_rate_last4,...,rsi_14_at_T1,macd_hist_at_T1,px_vs_sma20_T1,px_vs_sma10_T1,ret_14d,rsi_mean,rsi_slope,vol_mean,vol_ratio_mean,ret_std_14d
0,ABT,2022-01-26,1.32,1.21,2021-12-31,healthcare,0.11,1,0.473684,1.0,...,27.532126,-1.281694,-0.070017,-0.034338,-0.084753,38.516717,-21.898566,0.012201,1.236416,0.009417
1,ABT,2022-04-20,1.73,1.46,2022-03-31,healthcare,0.27,1,0.282297,1.0,...,51.414768,-0.031037,0.007278,0.003988,0.000650,49.120066,1.094245,0.015706,0.942901,0.016564
2,ABT,2022-07-20,1.43,1.14,2022-06-30,healthcare,0.29,1,0.249842,1.0,...,53.714133,0.282591,0.020945,0.018398,0.012086,48.178845,4.789194,0.017380,0.860467,0.015177
3,ABT,2022-10-19,1.15,0.94,2022-09-30,healthcare,0.21,1,0.250978,1.0,...,57.838722,0.604707,0.043616,0.029627,0.073103,46.292190,26.335915,0.015388,1.061622,0.018445
4,ABT,2023-01-25,1.03,0.94,2022-12-31,healthcare,0.09,1,0.188408,1.0,...,57.427281,-0.060655,0.012554,-0.001426,0.016451,62.244082,-5.529855,0.011614,1.182538,0.011669
